# HEART / Uni-Mol — best checkpoint embedding evaluation

**用途：**在 CMRI HPC Jupyter 中，唯讀載入舊訓練 `checkpoint_best.pt`，抽樣原有的 `valid.lmdb`，嘗試取得分子層級（CLS）embedding，產生 UMAP/PCA 與結構 descriptors 比對。**不會提交 SLURM 工作、不會改寫 checkpoint、不影響正在進行的 H200 training。**

**重要限制：**你的 `unimol_contrastive` 是自訂架構，尚未拿到 HPC 上實際的 `unimol/models/*.py`。以下 notebook 使用 Uni-Core task/model registry 及 forward hook；**若抓不到明確的 CLS representation，會停止並要求檢查 model forward，而不會將不明張量錯當 embeddings**。原始 pretraining validation pipeline 可能包含 masking / noise，因此此圖是 **pretraining-batch representation diagnostic**，還不是固定 conformer 的正式 downstream inference。

建議先在 HPC 上已安裝 `heart_gpu` 的 Jupyter kernel 執行；需要 `numpy`, `pandas`, `matplotlib`, `sklearn`，UMAP 為 optional。

In [ ]:
from pathlib import Path
import os, sys, re, json, inspect, warnings
import numpy as np
import torch

ROOT = Path('/home/ypan/heart_liver/model_train/heart-molecular-pretraining')
MODEL_ROOT = ROOT / 'model' / 'pretrain'
RUN_DIR = ROOT / 'runs' / 'full' / 'heart-pretrain-full-v1'  # OLD training, NOT H200
CKPT = RUN_DIR / 'checkpoint_best.pt'
DATA_DIR = Path('/home/ypan/heart_liver/model_train/UniMolReference/ligand_data/ligands')
UNICORE_ROOT = Path('/home/ypan/heart_liver/model_train/third_party/Uni-Core')
OUT = ROOT / 'runs' / 'evaluation' / 'old_best_embeddings'
OUT.mkdir(parents=True, exist_ok=True)
SEED = 17
MAX_MOLECULES = 192  # keep small; Jupyter CPU-friendly
BATCH_SIZE = 8
DEVICE = torch.device('cpu')  # avoids competing with H200 GPU; can switch only if separately allocated
np.random.seed(SEED); torch.manual_seed(SEED)
for p in (CKPT, DATA_DIR / 'valid.lmdb', MODEL_ROOT, UNICORE_ROOT):
    print(('OK' if p.exists() else 'MISSING'), p)
assert CKPT.is_file() and (DATA_DIR / 'valid.lmdb').exists(), 'Fix paths before continuing.'
print('Torch:', torch.__version__, '| Device:', DEVICE)


## A. Checkpoint metadata (no model execution)

In [ ]:
# Only load checkpoints you created/trust. weights_only=False uses pickle.
ck = torch.load(CKPT, map_location='cpu', weights_only=False)
print('Checkpoint keys:', list(ck.keys()))
args = ck.get('args')
if args is None:
    args = ck.get('cfg')
print('Argument object:', type(args).__name__)
print('Model state tensors:', len(ck.get('model', {})))
print('Parameter count:', sum(x.numel() for x in ck.get('model', {}).values() if torch.is_tensor(x)))
for key in ['extra_state','optimizer_history','best_loss','num_updates']:
    value = ck.get(key)
    print(key, str(value)[:450])
if args is not None:
    keys = vars(args) if hasattr(args,'__dict__') else (dict(args) if isinstance(args,dict) else {})
    print('Relevant args:', {k:str(v)[:150] for k,v in keys.items() if any(w in k for w in ('arch','task','loss','data','max_atoms','only_polar','seed','mask','noise','dict'))})


## B. Register custom Uni-Mol architecture and rebuild model

這一步需要 repository 中的自訂 `unimol_contrastive` registry。若失敗，先看 traceback（例如 CUDA extension / version mismatch），**不要載入官方 `unimol_tools` 預訓練權重來替代**。

In [ ]:
sys.path.insert(0, str(MODEL_ROOT))
sys.path.insert(0, str(UNICORE_ROOT))
os.chdir(MODEL_ROOT)

from unicore import tasks, utils
# Explicit import of custom module triggers task/model registration
import unimol

# Your checkpoint should contain argparse Namespace / OmegaConf-compatible cfg.
# First try Uni-Core standard task/model loading without changing weights.
if args is None:
    raise RuntimeError('Checkpoint has neither args nor cfg; inspect training entry and provide task arguments manually.')

# Prevent the dataset from being loaded from a stale path in checkpoint.
try:
    args.data = str(DATA_DIR)
    args.num_workers = 0
except Exception as e:
    print('Cannot change args directly:', repr(e))

task = tasks.setup_task(args)
model = task.build_model(args)
missing, unexpected = model.load_state_dict(ck['model'], strict=False)
print('Missing model keys:', len(missing), missing[:15])
print('Unexpected model keys:', len(unexpected), unexpected[:15])
if missing or unexpected:
    raise RuntimeError('Model and checkpoint are NOT an exact match. Do not interpret embeddings until resolved.')
model.to(DEVICE).eval()
print('Model type:', type(model))
print('Likely encoder modules:')
for name, module in model.named_modules():
    if name and (name.lower().endswith('encoder') or 'unimol' in name.lower()) and len(name.split('.')) <= 3:
        print(' ', name, type(module).__name__)
print('Forward signature:', inspect.signature(model.forward))


## C. Load a small *validation* batch using the same Uni-Core dataset pipeline

防止資料量過大，預設只分析前 192 筆 batch samples。這不是隨機抽取全部 150GB LMDB。建議第一輪先確認形狀和輸出；之後再擴大。

In [ ]:
task.load_dataset('valid', combine=False, epoch=1)
dataset = task.dataset('valid')
print('Validation dataset length:', len(dataset))
# Uni-Core standard batch iterator returns collated sample dictionaries.
itr = task.get_batch_iterator(
    dataset=dataset,
    max_sentences=BATCH_SIZE,
    max_positions=utils.resolve_max_positions(task.max_positions(), model.max_positions()),
    ignore_invalid_inputs=True,
    required_batch_size_multiple=1,
    seed=SEED,
    num_workers=0,
    epoch=1,
    disable_iterator_cache=True,
).next_epoch_itr(shuffle=False)
first = next(iter(itr))
print('Batch keys:', list(first))
print('net_input keys:', list(first.get('net_input', {})))
for k,v in first.get('net_input',{}).items():
    print('  ',k, tuple(v.shape) if torch.is_tensor(v) else type(v).__name__)


## D. Locate the real molecular representation

先試 `features_only=True`，並檢查回傳張量。若自訂 contrastive model 不支援，將在下一步用 encoder forward hook 搜集 token-level hidden states。**只接受 shape 為 `[batch, atoms/tokens, hidden_dim]` 的 tensor，取第一個 token (CLS)**。如果模型其實不使用 CLS，需要查看它的 pooling 定義再修改。

In [ ]:
def to_device(x, device):
    if torch.is_tensor(x): return x.to(device)
    if isinstance(x, dict): return {k:to_device(v,device) for k,v in x.items()}
    if isinstance(x, (list,tuple)): return type(x)(to_device(v,device) for v in x)
    return x

def tensor_shapes(obj, prefix='out', depth=0):
    if depth > 2: return []
    if torch.is_tensor(obj): return [(prefix, tuple(obj.shape))]
    if isinstance(obj,dict):
        return sum((tensor_shapes(v, f'{prefix}.{k}', depth+1) for k,v in obj.items()), [])
    if isinstance(obj,(tuple,list)):
        return sum((tensor_shapes(v, f'{prefix}[{i}]', depth+1) for i,v in enumerate(obj)), [])
    return []

net = to_device(first['net_input'], DEVICE)
with torch.inference_mode():
    try:
        preview = model(**net, features_only=True)
        print('features_only forward output:', tensor_shapes(preview))
    except Exception as exc:
        print('features_only is unavailable:', type(exc).__name__, str(exc)[:600])
        preview = None


## E. Hook the encoder, extract CLS embeddings

*Use only if C/D confirmed batch and module shapes.* The code prefers the deepest named encoder module producing 3D hidden states, and rejects ambiguous extraction. If no clear encoder exists, inspect the printed module names and alter `ENCODER_MODULE_NAME` manually. `FIRST_TOKEN_IS_CLS` must be confirmed from your task's tokenization (`<bos>` prepended).

In [ ]:
# Inspect candidate encoder modules before running many batches.
encoder_candidates = [(name,m) for name,m in model.named_modules()
                      if name and ('encoder' in name.lower()) and len(name.split('.')) <= 4]
print('Candidate names:', [x[0] for x in encoder_candidates[:35]])
# Override after inspecting source if necessary, e.g. 'encoder' or 'unimol.encoder'
ENCODER_MODULE_NAME = None
FIRST_TOKEN_IS_CLS = True  # standard Uni-Mol tokenization; verify in your custom task

# Diagnose 3D tensors captured by named encoder modules from ONE batch only.
collected = {}
handles=[]
def make_hook(nm):
    def hook(module, args, output):
        shapes=tensor_shapes(output, nm)
        matches=[]
        def walk(x):
            if torch.is_tensor(x) and x.ndim==3: matches.append(x.detach())
            elif isinstance(x,(tuple,list)):
                for v in x: walk(v)
            elif isinstance(x,dict):
                for v in x.values(): walk(v)
        walk(output)
        if matches: collected[nm] = [tuple(v.shape) for v in matches]
    return hook
for name, module in encoder_candidates:
    handles.append(module.register_forward_hook(make_hook(name)))
with torch.inference_mode():
    try:
        _ = model(**net)
    finally:
        for handle in handles: handle.remove()
print('Encoder outputs with rank-3 tensors:', json.dumps(collected, indent=2))


**選擇 hook 名稱：** 下方若只有一個 clearly molecule encoder，填入該名稱。若有多個 / view-specific representations，請先檢查 model code；**避免把 contrastive projection head / pair representations 當成 molecule embedding。**

In [ ]:
# Automatic selection is deliberately conservative.
if ENCODER_MODULE_NAME is None:
    choices = [name for name,shapes in collected.items()
               if any(len(s)==3 and s[0]==net[next(iter(net))].shape[0] for s in shapes)]
    if len(choices)==1:
        ENCODER_MODULE_NAME=choices[0]
    else:
        print('Ambiguous candidates:', choices)
        print('Set ENCODER_MODULE_NAME manually after checking model source, then rerun this cell.')
print('Selected:', ENCODER_MODULE_NAME)
assert ENCODER_MODULE_NAME is not None, 'Stop: no unambiguous encoder. Inspect source and set ENCODER_MODULE_NAME.'
selected_module=dict(model.named_modules())[ENCODER_MODULE_NAME]


In [ ]:
embeddings=[]; indices=[]
# Recreate validation iterator; first preview consumed an iterator.
iterator=task.get_batch_iterator(
    dataset=dataset, max_sentences=BATCH_SIZE,
    max_positions=utils.resolve_max_positions(task.max_positions(),model.max_positions()),
    ignore_invalid_inputs=True, required_batch_size_multiple=1,
    seed=SEED, num_workers=0, epoch=1, disable_iterator_cache=True
).next_epoch_itr(shuffle=False)

current=[]
def save_hidden(module, inputs, output):
    def find(x):
        if torch.is_tensor(x) and x.ndim==3: return x
        if isinstance(x,(tuple,list)):
            for item in x:
                z=find(item)
                if z is not None: return z
        if isinstance(x,dict):
            for item in x.values():
                z=find(item)
                if z is not None: return z
        return None
    hidden=find(output)
    if hidden is not None: current.append(hidden.detach().float().cpu())

h=selected_module.register_forward_hook(save_hidden)
count=0
try:
    with torch.inference_mode():
        for sample in iterator:
            current.clear()
            inp=to_device(sample['net_input'],DEVICE)
            _=model(**inp)
            if len(current)!=1:
                raise RuntimeError(f'Expected ONE encoder output per forward, got {len(current)}. The model may be multi-view; inspect forward.')
            token_hidden=current[0]
            bsz=sample['net_input'][next(iter(sample['net_input']))].shape[0]
            if token_hidden.ndim!=3 or token_hidden.shape[0]!=bsz:
                raise RuntimeError(f'Unexpected encoder shape {tuple(token_hidden.shape)} for batch {bsz}; confirm batch-first.')
            if not FIRST_TOKEN_IS_CLS:
                raise RuntimeError('CLS not confirmed; implement correct pooling based on model source.')
            arr=token_hidden[:,0,:].numpy()
            keep=min(len(arr),MAX_MOLECULES-count)
            embeddings.append(arr[:keep]); indices.extend(range(count,count+keep)); count+=keep
            if count>=MAX_MOLECULES: break
finally:
    h.remove()
X=np.concatenate(embeddings)
assert X.ndim==2 and X.shape[0]==count and np.isfinite(X).all()
print('Embedding matrix:', X.shape, '| variance median:', np.median(X.var(axis=0)))
if np.median(X.var(axis=0))<1e-9: warnings.warn('Embeddings near-constant; investigate collapse/wrong hook')
np.save(OUT/'best_valid_embeddings.npy',X)
np.save(OUT/'best_valid_row_order.npy',np.asarray(indices))
print('Saved:',OUT)


## F. PCA, UMAP, and nearest-neighbor sanity checks

沒有真實 mTP labels，因此不能把 UMAP 聚類解讀為高或低 delivery efficiency。此處按 embedding 階段的主成分著色，只檢查幾何結構與分布。

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

Xs=StandardScaler().fit_transform(X)
pca=PCA(n_components=2,random_state=SEED)
xy_pca=pca.fit_transform(Xs)
fig, ax=plt.subplots(figsize=(7,5))
sc=ax.scatter(xy_pca[:,0],xy_pca[:,1],c=np.arange(len(X)),s=20,cmap='viridis')
ax.set(title='HEART best.pt | Valid molecules (PCA)', xlabel='PC1',ylabel='PC2')
fig.colorbar(sc,ax=ax,label='Batch row order (NOT a biology label)')
fig.tight_layout(); fig.savefig(OUT/'pca.png',dpi=180); plt.show()
print('PC1/PC2 explained variance:',pca.explained_variance_ratio_)

try:
    import umap
    reducer=umap.UMAP(n_neighbors=min(15,len(X)-1),min_dist=.1,random_state=SEED)
    z=reducer.fit_transform(Xs)
    fig,ax=plt.subplots(figsize=(7,5))
    sc=ax.scatter(z[:,0],z[:,1],c=np.arange(len(z)),s=20,cmap='viridis')
    ax.set(title='HEART best.pt | Valid molecules (UMAP)',xlabel='UMAP1',ylabel='UMAP2')
    fig.colorbar(sc,ax=ax,label='Batch row order (NOT an mTP label)')
    fig.tight_layout();fig.savefig(OUT/'umap.png',dpi=180);plt.show()
    np.save(OUT/'umap_coordinates.npy',z)
except ImportError:
    print('umap-learn unavailable. Optional install in kernel env: pip install umap-learn')

s=cosine_similarity(X[:min(len(X),100)])
np.fill_diagonal(s,-np.inf)
print('Nearest neighbor similarity mean:',np.mean(s.max(axis=1)))
print('Embedding dimension std, min/median/max:',np.std(X,axis=0).min(),np.median(np.std(X,axis=0)),np.std(X,axis=0).max())


## G. Outcome and caveats

- Files: `runs/evaluation/old_best_embeddings/best_valid_embeddings.npy`, `pca.png`, optional `umap.png`.
- **No proof of chemical similarity or mTP prediction yet**: a chemically interpretable UMAP needs corresponding SMILES / physicochemical descriptors and a **stable, unmasked fixed-conformer inference pipeline**.
- On customized `unimol_contrastive` implementations, hooking a shared encoder may run it twice for two contrastive views; that is an intentional stop condition, not a failure of your checkpoint.
- If a cell fails, send me **the error traceback + printed candidate encoder names**. We can modify exactly the extraction cell to match the repository without changing the training code.
